# train_anomalib_pc (patched)

- 목표: **screw_bag, pushpins**만 대상으로 PatchCore 성능 개선 실험
- 제약: **임계값(Threshold)은 anomalib이 모델 학습 과정에서 계산/저장한 값을 그대로 사용**
- 실험: (1) Tiling (2) Light/Color Aug (3) Tiling+Aug


- 추가: **BASE(tiling/aug 없음)** 를 함께 돌리고, 각 실험의 개선량(Δ)을 표로 출력합니다.
- 지표는 기본적으로 **train_anomalib_base.py 기준(= pred_score > 0.5)** 으로 계산합니다. (model threshold 기반 지표는 참고용으로 함께 저장)


In [ ]:
# Cell 0) Drive mount + 경로 (robust)

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys

PROJECT_ROOT = Path("/content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation")
MMAD_ROOT    = Path("/content/drive/MyDrive/MMAD")
WORKDIR      = PROJECT_ROOT / "notebooks" / "jo"
CONFIG_PATH  = PROJECT_ROOT / "configs" / "anomaly.yaml"

WORKDIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT, "| exists:", PROJECT_ROOT.exists())
print("MMAD_ROOT   :", MMAD_ROOT,    "| exists:", MMAD_ROOT.exists())
print("WORKDIR     :", WORKDIR,      "| exists:", WORKDIR.exists())
print("CONFIG_PATH :", CONFIG_PATH,  "| exists:", CONFIG_PATH.exists())

assert PROJECT_ROOT.exists(), "PROJECT_ROOT 경로가 존재하지 않습니다."
assert MMAD_ROOT.exists(), "MMAD_ROOT 경로가 존재하지 않습니다."
assert CONFIG_PATH.exists(), "CONFIG_PATH(anomaly.yaml) 경로가 존재하지 않습니다."

# ✅ repo path 추가 (import 안정화)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ✅ subprocess/내부 모듈에서도 안정적으로 찾도록 PYTHONPATH도 세팅
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + (os.pathsep + os.environ["PYTHONPATH"] if "PYTHONPATH" in os.environ else "")

# ✅ 작업 디렉토리 이동
%cd "{PROJECT_ROOT}"

print("sys.path[0]:", sys.path[0])
print("PYTHONPATH:", os.environ.get("PYTHONPATH", "")[:200], "...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation | exists: True
MMAD_ROOT   : /content/drive/MyDrive/MMAD | exists: True
WORKDIR     : /content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation/notebooks/jo | exists: True
CONFIG_PATH : /content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation/configs/anomaly.yaml | exists: True
/content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation
sys.path[0]: /content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation
PYTHONPATH: /content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation:/env/python ...


In [ ]:
# Cell 1) 데이터/구조 빠른 체크 (경로 확인용)

print("MMAD ROOT listing (top):")
for p in sorted(MMAD_ROOT.glob("*"))[:20]:
    print(" -", p.name)

print("\nMVTec-LOCO exists:", (MMAD_ROOT / "MVTec-LOCO").exists())
print("GoodsAD exists   :", (MMAD_ROOT / "GoodsAD").exists())

MMAD ROOT listing (top):
 - DS-MVTec
 - GoodsAD
 - GoodsAD.zip
 - LICENSE-DATASET
 - MMAD_index.csv
 - MVTec-AD
 - MVTec-LOCO
 - MVTec-LOCO.zip
 - VisA
 - cfia_knowledge.json
 - cfia_knowledge_original.json
 - checkpoints
 - domain_knowledge.json
 - metadata.csv
 - mmad.json
 - mmad.jsonl
 - mmad_10classes.json
 - packaging_guide.pdf
 - winclip_llm_dataset.json

MVTec-LOCO exists: True
GoodsAD exists   : True


In [ ]:
# Cell 1) (옵션) 의존성 설치
# 이미 설치되어 있으면 생략해도 됩니다.
!pip -q install -r /content/drive/Othercomputers/내\ 노트북/multimodal-anomaly-report-generation/requirements.txt
!pip -q install anomalib opencv-python-headless scikit-learn pandas pillow tqdm


In [ ]:
# Cell 2) 런타임/출력 안정화 (RecursionError 방지용)
import os
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("RICH_DISABLE", "1")
os.environ.setdefault("RICH_NO_COLOR", "1")

# Lightning에서 model summary / progress bar가 출력될 때 rich-tqdm 충돌이 나면 아래 옵션들이 중요합니다.


'1'

In [ ]:
# Cell 3) 공통 import
import gc, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, roc_auc_score

from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.pre_processing import PreProcessor
from torchvision.transforms import Compose
from torchvision.transforms.v2 import (
    Resize, ToImage, ToDtype, ColorJitter, RandomAutocontrast, RandomEqualize, RandomAdjustSharpness, RandomGrayscale
)
from torchvision.transforms.v2 import Normalize as NormalizeV2

from src.utils.loaders import load_config
from src.utils.device import get_device
from src.datasets.dataloader import MMADLoader

try:
    from anomalib.callbacks import TilerConfigurationCallback
except Exception:
    TilerConfigurationCallback = None

def cleanup():
    gc.collect(); gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

def _to_float(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return float(x.detach().cpu().item())
    try:
        return float(x)
    except Exception:
        return None

def extract_image_threshold(model, fallback=None):
    thr_obj = getattr(model, "threshold", None)
    v = getattr(thr_obj, "value", None) if thr_obj is not None else None
    thr = _to_float(v)
    return fallback if thr is None else thr


In [ ]:
# Cell 4) 모델/transform 유틸 (baseline과 동일한 threshold 사용)
class NormalizeFlex(torch.nn.Module):
    """PreProcessor가 (image, mask) 형태로 호출돼도 안전."""
    def __init__(self, mean, std):
        super().__init__()
        self.norm = NormalizeV2(mean=mean, std=std)
    def forward(self, image, mask=None):
        image = self.norm(image)
        return image if mask is None else (image, mask)

def make_patchcore(cfg, overrides=None):
    """cfg(anomaly.yaml) 기반 PatchCore 생성.
    overrides: dict로 patchcore 하이퍼파라미터를 일부 덮어쓰고 싶을 때 사용(선택).
      - 지원하지 않는 키는 자동으로 무시(안전)
    """
    model_params = cfg["anomaly"].get("patchcore", {})

    # PreProcessor는 Normalize만 담당(Resize는 datamodule transform에서 수행)
    pre = PreProcessor(transform=NormalizeFlex(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225],
    ))

    # None 값 제거
    model_params = {k: v for k, v in model_params.items() if v is not None}

    # 선택 overrides 적용(지원 키만)
    if overrides:
        import inspect
        sig = inspect.signature(Patchcore.__init__)
        valid = set(sig.parameters.keys())
        for k, v in overrides.items():
            if k in valid:
                model_params[k] = v
            else:
                print(f"[skip override] Patchcore has no arg '{k}' (check anomalib version)")

    return Patchcore(pre_processor=pre, **model_params)

def build_transforms(image_size=(384, 384), use_light_color_aug=False):

    base = [
        ToImage(),
        Resize(image_size),
        ToDtype(torch.float32, scale=True),
    ]

    if use_light_color_aug:
        aug = [
            ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
            RandomAutocontrast(p=0.2),
            RandomEqualize(p=0.1),
            RandomAdjustSharpness(sharpness_factor=1.5, p=0.15),
            RandomGrayscale(p=0.05),
        ]
        train_tf = Compose(base + aug)
    else:
        train_tf = Compose(base)

    eval_tf = Compose(base)  # 평가/테스트는 항상 clean
    return train_tf, eval_tf

def inject_transforms(datamodule, train_tf, eval_tf):
    # datamodule field
    for name in ["train_transform", "val_transform", "test_transform", "eval_transform", "predict_transform"]:
        if hasattr(datamodule, name):
            setattr(datamodule, name, train_tf if "train" in name else eval_tf)

    # dataset field
    for attr in ["train_data","val_data","test_data","predict_data",
                 "train_dataset","val_dataset","test_dataset","predict_dataset"]:
        if hasattr(datamodule, attr):
            ds = getattr(datamodule, attr)
            if ds is None:
                continue
            if hasattr(ds, "transform"):
                ds.transform = train_tf if "train" in attr else eval_tf
            if hasattr(ds, "datasets"):
                for sub in ds.datasets:
                    if hasattr(sub, "transform"):
                        sub.transform = train_tf if "train" in attr else eval_tf
    return datamodule

def get_eval_loader(dm):
    # GT가 필요하므로 test_dataloader 우선
    if hasattr(dm, "test_dataloader"):
        tl = dm.test_dataloader()
        if tl is not None:
            return tl
    if hasattr(dm, "predict_dataloader"):
        pl = dm.predict_dataloader()
        if pl is not None:
            return pl
    if hasattr(dm, "val_dataloader"):
        vl = dm.val_dataloader()
        if vl is not None:
            return vl
    raise RuntimeError("test/predict/val dataloader가 모두 None 입니다. dm.setup(stage=...) 구성을 확인해 주세요.")


In [ ]:
# Cell 5) 단일 실험 러너 (tiling/aug 조합)
def compute_summary(df, label_col="pred_label"):
    """train_anomalib_base.py 기준(= pred_score>0.5로 만든 pred_label)과 비교 가능한 지표 요약.

    label_col:
      - 'pred_label'         : 현재 기준 라벨 (base 기준: pred_score>0.5)
      - 'pred_label_model'   : anomalib/model threshold 기반 라벨(참고용)
    """
    import numpy as np
    from sklearn.metrics import (
        confusion_matrix, roc_auc_score,
        precision_score, recall_score, f1_score,
        average_precision_score, balanced_accuracy_score,
        matthews_corrcoef,
    )

    df = df.dropna(subset=["gt_label", label_col])
    if len(df) == 0:
        return {
            "n": 0, "n_normal": 0, "n_anomaly": 0,
            "TN": 0, "FP": 0, "FN": 0, "TP": 0,
            "normal_acc": None, "fp_rate": None,
            "precision": None, "recall": None, "f1": None,
            "balanced_acc": None, "mcc": None,
            "image_auroc": None, "image_ap": None,
        }

    y_true = df["gt_label"].astype(int).to_numpy()
    y_pred = df[label_col].astype(int).to_numpy()

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    n_normal = int((y_true == 0).sum())
    n_anom = int((y_true == 1).sum())

    normal_acc = tn / (tn + fp + 1e-9)      # specificity
    fp_rate = fp / (tn + fp + 1e-9)         # FPR
    recall = tp / (tp + fn + 1e-9)          # TPR
    precision = tp / (tp + fp + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    # sklearn (방어적으로)
    try:
        precision_s = float(precision_score(y_true, y_pred, zero_division=0))
        recall_s = float(recall_score(y_true, y_pred, zero_division=0))
        f1_s = float(f1_score(y_true, y_pred, zero_division=0))
    except Exception:
        precision_s, recall_s, f1_s = float(precision), float(recall), float(f1)

    try:
        bacc = float(balanced_accuracy_score(y_true, y_pred))
    except Exception:
        bacc = None
    try:
        mcc = float(matthews_corrcoef(y_true, y_pred))
    except Exception:
        mcc = None

    # threshold-free(점수 기반) 지표
    auroc = None
    ap = None
    try:
        if df["pred_score"].notna().all() and len(np.unique(y_true)) > 1:
            scores = df["pred_score"].astype(float).to_numpy()
            auroc = float(roc_auc_score(y_true, scores))
            ap = float(average_precision_score(y_true, scores))
    except Exception:
        pass

    return {
        "n": int(len(df)),
        "n_normal": n_normal,
        "n_anomaly": n_anom,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "normal_acc": float(normal_acc),
        "fp_rate": float(fp_rate),
        "precision": precision_s,
        "recall": recall_s,
        "f1": f1_s,
        "balanced_acc": bacc,
        "mcc": mcc,
        "image_auroc": auroc,
        "image_ap": ap,
    }


def _safe_len(x):
    try:
        return len(x)
    except Exception:
        return None


def _expected_tiles(image_hw=(384, 384), tile=256, stride=128):
    """입력(384,384)에서 tile/stride로 몇 타일이 나올지 대략 계산(디버그용)."""
    H, W = image_hw
    if tile is None or stride is None:
        return None
    if tile <= 0 or stride <= 0:
        return None
    if tile > H or tile > W:
        return 0
    n_h = 1 + max(0, (H - tile) // stride)
    n_w = 1 + max(0, (W - tile) // stride)
    return int(n_h * n_w)


def _probe_tiler(model):
    """모델 내부에서 tiler가 어디에 붙는지 버전별로 달라서 최대한 넓게 탐색."""
    pp = getattr(model, "pre_processor", None)
    if pp is None:
        return None, "no pre_processor"

    # 1) 흔한 경로
    for name in ["tiler", "_tiler", "image_tiler"]:
        t = getattr(pp, name, None)
        if t is not None:
            return t, f"pre_processor.{name}"

    # 2) pre_processor 내부 속성들 중 'til' 포함하는 것 탐색
    for name in dir(pp):
        if "til" not in name.lower():
            continue
        try:
            obj = getattr(pp, name, None)
        except Exception:
            continue
        if obj is None:
            continue

        # obj 자체가 tiler처럼 보이는 경우
        if all(hasattr(obj, k) for k in ["tile_size", "stride"]):
            return obj, f"pre_processor.{name}"

        # obj 안에 tiler가 있는 경우
        if hasattr(obj, "tiler"):
            try:
                t = getattr(obj, "tiler", None)
                if t is not None:
                    return t, f"pre_processor.{name}.tiler"
            except Exception:
                pass

    return None, "not found"


def run_one(
    dataset, category,
    exp_name,
    use_tiling=False,
    use_light_color_aug=False,
    pc_overrides=None,
    tile=256, stride=128,
    train_bs=8, eval_bs=8, num_workers=2, max_epochs=1,
):

    import time
    import numpy as np
    import pandas as pd
    import torch

    device = get_device()

    # ----- config / model / datamodule -----
    cfg = load_config(str(CONFIG_PATH))
    cfg["data"]["root"] = str(MMAD_ROOT)

    loader = MMADLoader(config=cfg, model_name="patchcore")
    model = make_patchcore(cfg, overrides=pc_overrides)

    dm = loader.get_datamodule(
        dataset, category,
        train_batch_size=train_bs,
        eval_batch_size=eval_bs,
        num_workers=num_workers,
        include_mask=True,
    )

    # transforms: baseline resize 유지 + train에만 aug 옵션
    train_tf, eval_tf = build_transforms(image_size=(384, 384), use_light_color_aug=use_light_color_aug)
    dm = inject_transforms(dm, train_tf, eval_tf)

    # --- Sanity check (데이터 크기 / 배치 수) : setup(fit) 후에만 체크 ---
    try:
        try:
            dm.setup(stage="fit")
        except Exception:
            dm.setup()

        tr_dl = dm.train_dataloader()
        if tr_dl is None:
            print("[data] train_dataloader() is None (setup 타이밍/구현 영향) -> 학습 자체는 계속 진행")
        else:
            ds = getattr(tr_dl, "dataset", None)
            n_train = _safe_len(ds)
            n_batches = _safe_len(tr_dl)
            bs = getattr(tr_dl, "batch_size", None)
            print(f"[data] n_train={n_train} | n_batches={n_batches} | batch_size={bs}")
    except Exception as e:
        print("[data] train_dataloader sanity check failed:", repr(e))

    # ----- Engine / callbacks -----
    out_dir = WORKDIR / "pc_outputs" / exp_name / dataset / category
    callbacks = []

    if use_tiling:
        if TilerConfigurationCallback is None:
            raise RuntimeError("TilerConfigurationCallback import 실패: anomalib 버전을 확인해 주세요.")
        callbacks.append(TilerConfigurationCallback(enable=True, tile_size=tile, stride=stride))

    # ✅ fit 전에 콜백 실제로 들어갔는지 확인
    print("[callbacks]", [type(cb).__name__ for cb in callbacks])
    if use_tiling:
        exp_tiles = _expected_tiles((384, 384), tile=tile, stride=stride)
        print(f"[tiling:expected] image=384x384 | tile={tile} stride={stride} | expected_tiles_per_image≈{exp_tiles}")

    engine = Engine(
        accelerator="auto",
        devices=1,
        default_root_dir=str(out_dir),
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        max_epochs=max_epochs,
        callbacks=callbacks,
    )

    # ✅ trainer 콜백도 같이 찍어보기(엔진이 내부에서 추가/정리할 수 있음)
    try:
        print("[trainer.callbacks@init]", [type(cb).__name__ for cb in engine.trainer.callbacks])
    except Exception:
        pass

    # ----- fit -----
    t0 = time.time()
    engine.fit(model=model, datamodule=dm)
    train_sec = time.time() - t0

    # --- Sanity check (tiler 적용 여부) : 강제 탐색 ---
    try:
        tiler, where = _probe_tiler(model)
        if tiler is None:
            print(f"[tiler] None | where={where} (=> 실제로 tiling이 안 먹었거나, tiler가 다른 객체에 붙는 버전일 수 있음)")
        else:
            print(
                f"[tiler] FOUND at {where} | "
                f"enable={getattr(tiler,'enable',None)} | "
                f"tile_size={getattr(tiler,'tile_size',None)} | stride={getattr(tiler,'stride',None)}"
            )
    except Exception as e:
        print("[tiler] sanity check failed:", repr(e))

    thr_model = extract_image_threshold(model, fallback=0.5)

    # ----- eval -----
    model.eval().to(device)
    try:
        dm.setup(stage="test")
    except Exception:
        try:
            dm.setup(stage="predict")
        except Exception:
            dm.setup()

    eval_loader = get_eval_loader(dm)

    rows = []
    t1 = time.time()
    with torch.no_grad():
        for batch in eval_loader:
            images = batch.image.to(device)
            outputs = model(images)

            pred_score = getattr(outputs, "pred_score", None)
            pred_label_model_out = getattr(outputs, "pred_label", None)

            if pred_score is None:
                ps = np.full((images.shape[0],), np.nan)
                pl_base = np.full((images.shape[0],), np.nan)
                pl_model = np.full((images.shape[0],), np.nan)
            else:
                ps = pred_score.detach().cpu().numpy().reshape(-1)

                pl_base = (pred_score > 0.5).int().detach().cpu().numpy().reshape(-1)

                if pred_label_model_out is not None:
                    pl_model = pred_label_model_out.detach().cpu().numpy().reshape(-1)
                else:
                    pl_model = (pred_score > thr_model).int().detach().cpu().numpy().reshape(-1)

            gt = getattr(batch, "gt_label", None)
            gt = gt.detach().cpu().numpy().reshape(-1) if gt is not None else np.full((images.shape[0],), np.nan)

            paths = getattr(batch, "image_path", None)

            for i in range(images.shape[0]):
                rows.append(dict(
                    dataset=dataset, category=category, exp=exp_name,
                    gt_label=int(gt[i]) if not np.isnan(gt[i]) else None,
                    pred_score=float(ps[i]) if not np.isnan(ps[i]) else None,

                    # ✅ 비교용 (base 기준)
                    pred_label=int(pl_base[i]) if not np.isnan(pl_base[i]) else None,


                    pred_label_model=int(pl_model[i]) if not np.isnan(pl_model[i]) else None,

                    model_threshold=float(thr_model),
                    use_tiling=bool(use_tiling),
                    use_light_color_aug=bool(use_light_color_aug),
                    tile=int(tile) if use_tiling else None,
                    stride=int(stride) if use_tiling else None,
                    image_path=(paths[i] if isinstance(paths, (list, tuple)) else paths),
                ))

    eval_sec = time.time() - t1
    df_scores = pd.DataFrame(rows)

    # ✅ 요약 지표는 기본적으로 'pred_label'(=0.5 기준)로 계산 → 베이스라인 비교 가능
    summ = compute_summary(df_scores, label_col="pred_label")

    # 참고로 model-thr 라벨 지표도 같이 저장(디버깅용)
    summ_model = compute_summary(df_scores, label_col="pred_label_model")

    result = dict(
        dataset=dataset, category=category, exp=exp_name,
        train_sec=float(train_sec),
        eval_sec=float(eval_sec),
        num_images=int(len(df_scores)),

        model_threshold=float(thr_model),

        # base 기준 지표
        **summ,

        # model-thr 지표(접미사)
        normal_acc_model=float(summ_model["normal_acc"]) if summ_model.get("normal_acc") is not None else None,
        fp_rate_model=float(summ_model["fp_rate"]) if summ_model.get("fp_rate") is not None else None,
        f1_model=float(summ_model["f1"]) if summ_model.get("f1") is not None else None,
        precision_model=float(summ_model["precision"]) if summ_model.get("precision") is not None else None,
        recall_model=float(summ_model["recall"]) if summ_model.get("recall") is not None else None,
    )

    cleanup()
    return result, df_scores

In [ ]:
# Cell 6) 4가지 실험 실행 + 베이스라인 대비 개선량 계산 (screw_bag / pushpins)
import time

BUDGET_HOURS = 5
budget_sec = BUDGET_HOURS * 3600
t_all = time.time()

TARGETS = [
    ("MVTec-LOCO", "screw_bag"),
    ("MVTec-LOCO", "pushpins"),
]

EXPS = [
    # 0) Baseline
    dict(exp_name="BASE", use_tiling=False, use_light_color_aug=False),

    #타일링+색보정
    dict(exp_name="E1_tiling_256_128",           use_tiling=True,  use_light_color_aug=False, tile=256, stride=128),
    dict(exp_name="E2_lightcolor",               use_tiling=False, use_light_color_aug=True),
    dict(exp_name="E3_tiling256_128_lightcolor", use_tiling=True,  use_light_color_aug=True,  tile=256, stride=128),

    # 2) 타일 파라미터 2개만 추가 (성능/시간 트레이드오프 탐색)
    dict(exp_name="E4_tiling192_96_lightcolor",  use_tiling=True,  use_light_color_aug=True,  tile=192, stride=96),
    dict(exp_name="E5_tiling320_160_lightcolor", use_tiling=True,  use_light_color_aug=True,  tile=320, stride=160),

    # 3) FP 줄이기 후보: kNN 이웃수/코어셋 비율(지원되는 키만 적용됨)
    dict(exp_name="E6_neighbors15",              use_tiling=False, use_light_color_aug=False,
         pc_overrides={"num_neighbors": 15, "n_neighbors": 15}),
    dict(exp_name="E7_coreset20pct",             use_tiling=False, use_light_color_aug=False,
         pc_overrides={"coreset_sampling_ratio": 0.20, "coreset_percentage": 0.20}),
]

all_rows = []
all_details = []
run_times = []

stop = False
for dataset, category in TARGETS:
    if stop:
        break
    for exp in EXPS:
        elapsed = time.time() - t_all
        left = budget_sec - elapsed
        if left <= 0:
            print(f"⏱️ 시간 예산 {BUDGET_HOURS}시간 초과 → 이후 실험 중단 (부분 결과 저장)")
            stop = True
            break

        print(f"\n===== {dataset}/{category} :: {exp['exp_name']} =====")
        t0 = time.time()

        res, df_detail = run_one(
            dataset=dataset, category=category,
            exp_name=exp["exp_name"],
            use_tiling=exp.get("use_tiling", False),
            tile=exp.get("tile", 256),
            stride=exp.get("stride", 128),
            use_light_color_aug=exp.get("use_light_color_aug", False),
            pc_overrides=exp.get("pc_overrides", None),
        )

        dt = time.time() - t0
        run_times.append(dt)

        # 저장
        all_rows.append(res)
        all_details.append(df_detail)

        # 간단 진행상황
        elapsed = time.time() - t_all
        left = max(0, budget_sec - elapsed)
        mean_run = sum(run_times) / max(1, len(run_times))
        remaining_runs_est = (len(TARGETS) * len(EXPS)) - len(run_times)
        est_left = remaining_runs_est * mean_run
        print(f"⏱️ 이번 run: {dt/60:.1f} min | 누적: {elapsed/3600:.2f} h | 예산 잔여: {left/3600:.2f} h | 남은 run 대략: {remaining_runs_est}개 (추정 {est_left/3600:.2f} h)")

# 결과 합치기/저장
df_results = pd.DataFrame(all_rows)
df_details = pd.concat(all_details, ignore_index=True) if len(all_details) else pd.DataFrame()

SAVE_SUMMARY = WORKDIR / "results_pc_targets.csv"
SAVE_DETAILS = WORKDIR / "results_pc_targets_details.csv"
df_results.to_csv(SAVE_SUMMARY, index=False)
df_details.to_csv(SAVE_DETAILS, index=False)

print("\n✅ saved:", SAVE_SUMMARY)
print("✅ saved:", SAVE_DETAILS)

# 베이스라인 대비 개선량(Δ) 표
if not df_results.empty and (df_results["exp"] == "BASE").any():
    base = df_results[df_results["exp"] == "BASE"][["dataset","category","normal_acc","fp_rate","recall","precision","f1","image_auroc","image_ap"]]
    base = base.rename(columns={c: f"BASE_{c}" for c in base.columns if c not in ["dataset","category"]})

    merged = df_results.merge(base, on=["dataset","category"], how="left")

    # Δ 정의: 개선이 +가 되도록 방향 맞춤
    merged["d_normal_acc"] = merged["normal_acc"] - merged["BASE_normal_acc"]           # ↑ 좋음
    merged["d_fp_rate"]    = merged["BASE_fp_rate"] - merged["fp_rate"]               # ↓ 좋음 (베이스 - 실험)
    merged["d_f1"]         = merged["f1"] - merged["BASE_f1"]                         # ↑ 좋음
    merged["d_auroc"]      = merged["image_auroc"] - merged["BASE_image_auroc"]        # ↑ 좋음
    merged["d_ap"]         = merged["image_ap"] - merged["BASE_image_ap"]              # ↑ 좋음

    cols = ["dataset","category","exp",
            "normal_acc","fp_rate","recall","precision","f1","image_auroc","image_ap",
            "d_normal_acc","d_fp_rate","d_f1","d_auroc","d_ap",
            "train_sec","eval_sec","num_images"]
    cols = [c for c in cols if c in merged.columns]

    display(merged[cols].sort_values(["dataset","category","exp"]))



===== MVTec-LOCO/screw_bag :: BASE =====
Device: NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
INFO:lightning_fabric.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:lightning_fabric.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:lightning_fabric.utilities.rank_zero

[data] n_train=360 | n_batches=45 | batch_size=8
[callbacks] []


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:534: Found 174 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.
INFO:lightning_fabric.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


[tiler] None | where=not found (=> 실제로 tiling이 안 먹었거나, tiler가 다른 객체에 붙는 버전일 수 있음)
⏱️ 이번 run: 4.8 min | 누적: 0.08 h | 예산 잔여: 4.92 h | 남은 run 대략: 15개 (추정 1.20 h)

===== MVTec-LOCO/screw_bag :: E1_tiling_256_128 =====
Device: NVIDIA A100-SXM4-40GB


...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
INFO:lightning_fabric.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:lightning_fabric.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:lightning_fabric.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[data] n_train=372 | n_batches=47 | batch_size=8
[callbacks] []


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:534: Found 174 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.
INFO:lightning_fabric.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


[tiler] None | where=not found (=> 실제로 tiling이 안 먹었거나, tiler가 다른 객체에 붙는 버전일 수 있음)
⏱️ 이번 run: 9.0 min | 누적: 1.65 h | 예산 잔여: 3.35 h | 남은 run 대략: 0개 (추정 0.00 h)

✅ saved: /content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation/notebooks/jo/results_pc_targets.csv
✅ saved: /content/drive/Othercomputers/내 노트북/multimodal-anomaly-report-generation/notebooks/jo/results_pc_targets_details.csv


,dataset,category,exp,normal_acc,fp_rate,recall,precision,f1,image_auroc,image_ap,d_normal_acc,d_fp_rate,d_f1,d_auroc,d_ap,train_sec,eval_sec,num_images
8,MVTec-LOCO,pushpins,BASE,0.528986,0.471014,0.802326,0.679803,0.736000,0.773045,0.846411,0.000000,0.000000,0.000000,0.000000,0.000000,394.393119,20.970618,310
9,MVTec-LOCO,pushpins,E1_tiling_256_128,0.514493,0.485507,0.831395,0.680952,0.748691,0.773593,0.842777,-0.014493,-0.014493,0.012691,0.000548,-0.003634,271.145616,21.409373,310
10,MVTec-LOCO,pushpins,E2_lightcolor,0.550725,0.449275,0.813953,0.693069,0.748663,0.772835,0.846367,0.021739,0.021739,0.012663,-0.000211,-0.000043,271.857367,21.386867,310
11,MVTec-LOCO,pushpins,E3_tiling256_128_lightcolor,0.557971,0.442029,0.808140,0.695000,0.747312,0.768411,0.842156,0.028986,0.028986,0.011312,-0.004634,-0.004255,270.743151,20.124327,310
12,MVTec-LOCO,pushpins,E4_tiling192_96_lightcolor,0.478261,0.521739,0.819767,0.661972,0.732468,0.757668,0.838351,-0.050725,-0.050725,-0.003532,-0.015377,-0.008060,270.767590,21.159140,310
13,MVTec-LOCO,pushpins,E5_tiling320_160_lightcolor,0.420290,0.579710,0.843023,0.644444,0.730479,0.766389,0.845736,-0.108696,-0.108696,-0.005521,-0.006657,-0.000675,619.422068,32.796039,310
14,MVTec-LOCO,pushpins,E6_neighbors15,0.492754,0.507246,0.843023,0.674419,0.749354,0.788170,0.856066,-0.036232,-0.036232,0.013354,0.015125,0.009655,271.551091,20.631922,310
15,MVTec-LOCO,pushpins,E7_coreset20pct,0.514493,0.485507,0.808140,0.674757,0.735450,0.775784,0.850341,-0.014493,-0.014493,-0.000550,0.002738,0.003930,510.947256,26.122234,310
0,MVTec-LOCO,screw_bag,BASE,0.032787,0.967213,0.990868,0.647761,0.783394,0.707351,0.836978,0.000000,0.000000,0.000000,0.000000,0.000000,259.819035,23.038272,341
1,MVTec-LOCO,screw_bag,E1_tiling_256_128,0.057377,0.942623,0.990868,0.653614,0.787659,0.709297,0.836440,0.024590,0.024590,0.004265,0.001946,-0.000538,257.850880,23.211637,341
